In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv")
df.head()

df['Children in HHS Care']=df['Children in HHS Care'].str.replace(',','').astype(int)
df["Date"] = pd.to_datetime(df["Date"])
df.info()

df=df.sort_values('Date')
df=df.set_index('Date')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Date                                             720 non-null    datetime64[ns]
 1   Children apprehended and placed in CBP custody*  720 non-null    int64         
 2   Children in CBP custody                          720 non-null    int64         
 3   Children transferred out of CBP custody          720 non-null    int64         
 4   Children in HHS Care                             720 non-null    int64         
 5   Children discharged from HHS Care                720 non-null    int64         
dtypes: datetime64[ns](1), int64(5)
memory usage: 33.9 KB


In [3]:
'''Feature Engineering begins here'''
df["HHS_Lag_1"] = df["Children in HHS Care"].shift(1)
df["HHS_Lag_7"] = df["Children in HHS Care"].shift(7)
df["HHS_Lag_14"] = df["Children in HHS Care"].shift(14)

df["Rolling_Mean_7"] = (
    df["Children in HHS Care"]
    .rolling(window=7)
    .mean()
)

df["Rolling_Mean_14"] = (
    df["Children in HHS Care"]
    .rolling(window=14)
    .mean()
)

df["Rolling_STD_7"] = (
    df["Children in HHS Care"]
    .rolling(window=7)
    .std()
)

df["Net_Pressure"] = (
    df["Children transferred out of CBP custody"]
    - df["Children discharged from HHS Care"]
)

df["Year"] = df.index.year
df["Month"] = df.index.month
df["Day"] = df.index.day
df["Day_of_Week"] = df.index.dayofweek
df["Quarter"] = df.index.quarter
df["Is_Weekend"] = (df.index.dayofweek >= 5).astype(int)
df.head(5)

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,HHS_Lag_1,HHS_Lag_7,HHS_Lag_14,Rolling_Mean_7,Rolling_Mean_14,Rolling_STD_7,Net_Pressure,Year,Month,Day,Day_of_Week,Quarter,Is_Weekend
Date,,,,,,,,,,,,,,,,,,
2023-01-12,33,53,34,6566,436,NaN,NaN,NaN,NaN,NaN,NaN,-402,2023,1,12,3,1,0
2023-01-22,32,49,39,7122,227,6566.0,NaN,NaN,NaN,NaN,NaN,-188,2023,1,22,6,1,1
2023-01-23,32,50,39,7280,181,7122.0,NaN,NaN,NaN,NaN,NaN,-142,2023,1,23,0,1,0
2023-01-24,47,42,47,7433,175,7280.0,NaN,NaN,NaN,NaN,NaN,-128,2023,1,24,1,1,0
2023-01-25,20,22,41,7538,180,7433.0,NaN,NaN,NaN,NaN,NaN,-139,2023,1,25,2,1,0


In [4]:
df.isnull().sum()

Children apprehended and placed in CBP custody*     0
Children in CBP custody                             0
Children transferred out of CBP custody             0
Children in HHS Care                                0
Children discharged from HHS Care                   0
HHS_Lag_1                                           1
HHS_Lag_7                                           7
HHS_Lag_14                                         14
Rolling_Mean_7                                      6
Rolling_Mean_14                                    13
Rolling_STD_7                                       6
Net_Pressure                                        0
Year                                                0
Month                                               0
Day                                                 0
Day_of_Week                                         0
Quarter                                             0
Is_Weekend                                          0
dtype: int64

In [5]:
df.dropna(inplace=True)
df.head(5)

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care,HHS_Lag_1,HHS_Lag_7,HHS_Lag_14,Rolling_Mean_7,Rolling_Mean_14,Rolling_STD_7,Net_Pressure,Year,Month,Day,Day_of_Week,Quarter,Is_Weekend
Date,,,,,,,,,,,,,,,,,,
2023-02-09,124,234,161,7908,353,7915.0,7803.0,6566.0,7823.714286,7654.071429,124.514677,-192,2023,2,9,3,1,0
2023-02-12,92,203,173,7434,317,7908.0,7903.0,7122.0,7756.714286,7676.357143,185.827621,-144,2023,2,12,6,1,1
2023-02-13,186,259,172,7483,244,7434.0,7879.0,7280.0,7700.142857,7690.857143,201.971474,-72,2023,2,13,0,1,0
2023-02-14,154,225,220,7794,223,7483.0,7586.0,7433.0,7729.857143,7716.642857,197.633861,-3,2023,2,14,1,1,0
2023-02-15,91,199,172,7869,290,7794.0,7720.0,7538.0,7751.142857,7740.285714,204.306492,-118,2023,2,15,2,1,0


In [6]:
date_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="D"
)

print("Total expected dates:", len(date_range))
print("Total available dates:", len(df))
print("Missing dates:", len(date_range) - len(df))

Total expected dates: 1047
Total available dates: 706
Missing dates: 341


In [ ]:
date_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="D"
)

missing_dates = date_range.difference(df.index)

print(missing_dates)
print("Number of missing dates:", len(missing_dates))
df.to_csv("../data/processed/HHS_Feature_Engineered.csv")
df.to_csv("../data/processed/HHS_Clean_Dataset.csv")

DatetimeIndex(['2023-02-10', '2023-02-11', '2023-02-17', '2023-02-18',
               '2023-02-19', '2023-02-24', '2023-02-25', '2023-03-03',
               '2023-03-04', '2023-03-05',
               ...
               '2025-11-22', '2025-11-26', '2025-11-28', '2025-11-29',
               '2025-12-05', '2025-12-06', '2025-12-12', '2025-12-13',
               '2025-12-19', '2025-12-20'],
              dtype='datetime64[ns]', length=341, freq=None)
Number of missing dates: 341
